In [ ]:
import os
os.environ["QT_API"] = "pyqt5"
%matplotlib qt
import numpy as np
from matplotlib import pyplot as plt

from mne import Epochs, create_info
from mne.io import RawArray
from mne.time_frequency import  tfr_array_morlet
import pandas as pd
import numpy as np
import mne
import scipy.io as sio
import numpy as np
import glob
import seaborn as sns
import matplotlib.pyplot as plt
from mne.report import Report
import sys
from mne.preprocessing import ICA, create_eog_epochs, create_ecg_epochs, corrmap, read_ica
from mne.report import Report
import math 
import pickle
import mne
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import os
from mne.preprocessing import ICA, create_eog_epochs, create_ecg_epochs, corrmap
from mne.preprocessing import annotate_movement, compute_average_dev_head_t
from mne.report import Report
import math 
import pickle
from mne.time_frequency import tfr_array_multitaper, tfr_array_morlet, tfr_multitaper
import mne
from mne.channels import make_1020_channel_selections
from mne.event import define_target_events

In [ ]:
path="F:/epochs_ITC"
subjects = ["sub_m_1_02"]
for subj in subjects:
    epochs = mne.read_epochs(f"F:/epochs_ITC/{subj}-raw-ica-reject-ERP-epo.fif", preload = True)
print(epochs.event_id)

In [ ]:
epochs.times
len(epochs.times)
epochs.tmin, epochs.tmax

In [ ]:
#cluster-based permutation across 64 channel with topo plot-power for cue1
import numpy as np
import mne
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import os
os.environ["QT_API"] = "pyqt5"
%matplotlib qt

subjects = ["sub_m_1_02", "sub_m_1_06", "sub_m_1_10","sub_m_1_14","sub_m_1_18","sub_m_1_22","sub_m_1_26","sub_m_1_30","sub_m_1_34","sub_m_1_38","sub_m_1_42","sub_m_1_46",
            "sub_m_2_04","sub_m_2_08","sub_m_2_12","sub_m_2_16","sub_m_2_20","sub_m_2_24","sub_m_2_28","sub_m_2_32","sub_m_2_36","sub_m_2_40","sub_m_2_44",
            "sub_p_1_01","sub_p_1_05","sub_p_1_09","sub_p_1_13","sub_p_1_17","sub_p_1_21","sub_p_1_25","sub_p_1_29","sub_p_1_33","sub_p_1_35","sub_p_1_37","sub_p_1_41","sub_p_1_45",
            "sub_p_2_03","sub_p_2_07","sub_p_2_11","sub_p_2_15","sub_p_2_19","sub_p_2_27","sub_p_2_31","sub_p_2_39","sub_p_2_43","sub_p_2_47"]

OUT_DIR = r"F:/tfr_single_trial_cycle_8"
EPOCHS_DIR = r"F:/epochs_ITC"

conds = {"int_rep": "cue/cue1/int/rep", "int_swi": "cue/cue1/int/swi", "ext_rep": "cue/cue1/ext/rep","ext_swi": "cue/cue1/ext/swi"}

bands = {"Low Beta": (13, 16),"Beta": (16, 28),"alpha": (8, 15),"theta": (4, 7)}

BIN = 0.25
N_PERM = 2000
ALPHA = 0.05
TAIL = 0
SEED = 42

def safe_cond_indices(epochs, cond):
    cond_sel = epochs[cond].selection
    idx = np.flatnonzero(np.isin(epochs.selection, cond_sel))
    return idx

def cond_mean(X, epochs, cond):
    idx = safe_cond_indices(epochs, cond)
    if len(idx) == 0:
        raise RuntimeError(f"No trials for {cond}")
    return X[idx].mean(axis=0)  # (ch, freq, time)

def band_average(M, freqs, f_lo, f_hi):
    fmask = (freqs >= f_lo) & (freqs <= f_hi)
    if not np.any(fmask):
        raise RuntimeError(f"No freq bins in [{f_lo}, {f_hi}]")
    return M[:, fmask, :].mean(axis=1)  # (ch, time)

def build_subject_effects(subj, freqs_ref=None, times_ref=None, ch_ref=None):
    power_path = os.path.join(OUT_DIR, f"{subj}_power.npy")
    meta_path  = os.path.join(OUT_DIR, f"{subj}_meta.npz")
    epo_path   = os.path.join(EPOCHS_DIR, f"{subj}-raw-ica-reject-ERP-epo.fif")

    if not (os.path.exists(power_path) and os.path.exists(meta_path) and os.path.exists(epo_path)):
        return None, None, None, f"missing file(s)"

    X = np.load(power_path, mmap_mode="r")  # (trial, ch, freq, time)
    meta = np.load(meta_path, allow_pickle=True)
    ch_names = meta["ch_names"].tolist()
    freqs = meta["freqs"]
    times = meta["times"]

    # axis consistency check
    if freqs_ref is not None:
        if len(freqs) != len(freqs_ref) or not np.allclose(freqs, freqs_ref):
            return None, None, None, "freq mismatch"
    if times_ref is not None:
        if len(times) != len(times_ref) or not np.allclose(times, times_ref):
            return None, None, None, "time mismatch"
    if ch_ref is not None:
        if ch_names != ch_ref:
            return None, None, None, "channel order mismatch"

    # epochs: only used for condition trial indexing
    epochs = mne.read_epochs(epo_path, preload=True).crop(tmin=times[0], tmax=times[-1])

    if X.shape[0] != len(epochs):
        return None, None, None, f"trial count mismatch: X={X.shape[0]} vs epochs={len(epochs)}"

    # condition means: (ch,freq,time)
    M = {k: cond_mean(X, epochs, c) for k, c in conds.items()}

    # effects in full TF (ch,freq,time)
    INT = 0.5 * (M["int_rep"] + M["int_swi"])
    EXT = 0.5 * (M["ext_rep"] + M["ext_swi"])
    D_int_ext = INT - EXT

    SWI = 0.5 * (M["int_swi"] + M["ext_swi"])
    REP = 0.5 * (M["int_rep"] + M["ext_rep"])
    D_swi_rep = SWI - REP

    D_inter = (M["int_swi"] - M["int_rep"]) - (M["ext_swi"] - M["ext_rep"])

    out = {
        "rule_switch_int_ext": D_int_ext,
        "att_switch_swi_rep": D_swi_rep,
        "interaction": D_inter,
    }
    return out, freqs, times, None

def make_time_bins(times, bin_s=0.25):
    t0, t1 = float(times[0]), float(times[-1])
    edges = np.arange(t0, t1 + 1e-9, bin_s)
    if edges[-1] < t1:
        edges = np.append(edges, t1)
    bins = []
    for a, b in zip(edges[:-1], edges[1:]):
        mask = (times >= a) & (times < b) if b < t1 else (times >= a) & (times <= b)
        if mask.any():
            bins.append((a, b, mask))
    return bins

def run_cluster_time_ch(X_subj_ch_time, info, adjacency, alpha=0.05, n_perm=2000, tail=0, seed=42):
    # MNE expects (n_samples, n_times, n_space) for spatio-temporal with adjacency over space
    X_st = np.transpose(X_subj_ch_time, (0, 2, 1))  # -> (subj, time, ch)

    T_obs, clusters, pvals, H0 = mne.stats.spatio_temporal_cluster_1samp_test(
        X_st,
        adjacency=adjacency,
        n_permutations=n_perm,
        threshold=None,   # uses t-threshold internally (like your logs showed ~2.01)
        tail=tail,
        out_type="mask",
        seed=seed,
        verbose=True,
    )

    # Build sig mask over (time,ch)
    sig = np.zeros(T_obs.shape, dtype=bool)  # (time,ch)
    for cl, p in zip(clusters, pvals):
        if p < alpha:
            sig |= cl  # cl is boolean mask if out_type="mask"

    # convert to (ch,time) for topomap usage
    sig_ch_time = sig.T  # (ch,time)
    return T_obs, clusters, pvals, sig_ch_time

# 1) Load one subject to anchor info/adjacency + axes
anchor = subjects[0]
epo_anchor = os.path.join(EPOCHS_DIR, f"{anchor}-raw-ica-reject-ERP-epo.fif")
epochs_anchor = mne.read_epochs(epo_anchor, preload=True)
info = epochs_anchor.copy().pick_types(eeg=True).info

adjacency, ch_names_adj = mne.channels.find_ch_adjacency(info, ch_type="eeg")
print("[adjacency] shape:", adjacency.shape)

# 2) Build subject stacks for each effect+band
effects = ["att_switch_swi_rep", "rule_switch_int_ext", "interaction"]

# store: band -> effect -> list of (ch,time)
stack = {band: {eff: [] for eff in effects} for band in bands.keys()}

freqs_ref = None
times_ref = None
ch_ref = None
kept = 0

for subj in subjects:
    out, freqs, times, err = build_subject_effects(subj, freqs_ref, times_ref, ch_ref)
    if err is not None:
        print("SKIP", subj, "->", err)
        continue

    # lock references on first kept subject
    if freqs_ref is None:
        freqs_ref = freqs.copy()
        times_ref = times.copy()
        # channel names from your meta (must match X channel order)
        meta0 = np.load(os.path.join(OUT_DIR, f"{subj}_meta.npz"), allow_pickle=True)
        ch_ref = meta0["ch_names"].tolist()

    # band-average and collect
    for band_name, (f_lo, f_hi) in bands.items():
        for eff in effects:
            D_ch_f_t = out[eff]  # (ch,freq,time)
            D_ch_t = band_average(D_ch_f_t, freqs_ref, f_lo, f_hi)  # (ch,time)
            stack[band_name][eff].append(D_ch_t.astype(np.float32))

    kept += 1

print("Kept subjects:", kept)

# 3) Cluster + plot: many topomaps per time-bin with black dots
time_bins = make_time_bins(times_ref, BIN)
print("n_time_bins:", len(time_bins), "bin_s:", BIN)

for band_name, (f_lo, f_hi) in bands.items():
    all_maps = []
    for eff in effects:
        X_eff = np.stack(stack[band_name][eff], axis=0)  
        all_maps.append(X_eff.mean(axis=0))           
    all_maps = np.concatenate(all_maps, axis=1)
    vmin, vmax = np.percentile(all_maps, [5, 95])

    n_rows = len(effects)
    n_cols = len(time_bins)

    fig = plt.figure(figsize=(18, 10))
    fig.suptitle(
        f"{band_name.upper()} ({f_lo:.0f}-{f_hi:.0f} Hz) — Cluster permutation topomaps",
        y=0.98
    )

    # +1 column reserved for colorbar
    gs = gridspec.GridSpec(
        n_rows,
        n_cols + 1,
        width_ratios=[1] * n_cols + [0.05],
        wspace=0.25,
        hspace=0.3
    )

    #plot
    for r, eff in enumerate(effects):
        X_eff = np.stack(stack[band_name][eff], axis=0)  # (subj,ch,time)

        T_obs, clusters, pvals, sig_ch_time = run_cluster_time_ch(
            X_eff, info, adjacency,
            alpha=ALPHA, n_perm=N_PERM, tail=TAIL, seed=SEED
        )

        mean_ch_t = X_eff.mean(axis=0)  # (ch,time)

        for c, (ta, tb, tmask) in enumerate(time_bins):
            ax = fig.add_subplot(gs[r, c])

            dat = mean_ch_t[:, tmask].mean(axis=1)
            sig_ch = sig_ch_time[:, tmask].any(axis=1)

            mne.viz.plot_topomap(
                dat,
                info,
                axes=ax,
                show=False,
                vlim=(vmin, vmax),
                contours=0,
                sensors=False,
                mask=sig_ch,
                mask_params=dict(
                    markersize=6,
                    markerfacecolor="k",
                    markeredgecolor="k"
                )
            )

            ax.set_title(f"{ta:+.2f}–{tb:+.2f}s", fontsize=8)
            if c == 0:
                ax.set_ylabel(eff, fontsize=10)

    #  colorbar (single, dedicated axis) 
    cax = fig.add_subplot(gs[:, -1])
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="RdBu_r")
    sm.set_array([])

    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label("Power (baseline-corrected logratio)", rotation=90)

    plt.show()

In [ ]:
#Plot the list of the siginificant channels
#Significant channels across the time window
sig_ch = sig_ch_time[:, tmask].any(axis=1)
sig_names = np.array(info['ch_names'])[sig_ch]

print(
    f"[{band_name} | {eff} | {ta:+.2f}–{tb:+.2f}s] "
    f"{len(sig_names)} channels: {sig_names.tolist()}"
)

#significant cluster
best_idx = np.argmin(pvals)
best_cluster = clusters[best_idx]  # (time, ch) boolean

best_ch = best_cluster.any(axis=0)
best_ch_names = np.array(info['ch_names'])[best_ch]

print(
    f"[{band_name} | {eff}] strongest cluster p={pvals[best_idx]:.4f}"
)
print("Channels:", best_ch_names.tolist())

In [ ]:
import numpy as np
import mne
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import os
os.environ["QT_API"] = "pyqt5"
%matplotlib qt

subjects = ["sub_m_1_02", "sub_m_1_06", "sub_m_1_10","sub_m_1_14","sub_m_1_18","sub_m_1_22","sub_m_1_26","sub_m_1_30","sub_m_1_34","sub_m_1_38","sub_m_1_42","sub_m_1_46",
            "sub_m_2_04","sub_m_2_08","sub_m_2_12","sub_m_2_16","sub_m_2_20","sub_m_2_24","sub_m_2_28","sub_m_2_32","sub_m_2_36","sub_m_2_40","sub_m_2_44",
            "sub_p_1_01","sub_p_1_05","sub_p_1_09","sub_p_1_13","sub_p_1_17","sub_p_1_21","sub_p_1_25","sub_p_1_29","sub_p_1_33","sub_p_1_35","sub_p_1_37","sub_p_1_41","sub_p_1_45",
            "sub_p_2_03","sub_p_2_07","sub_p_2_11","sub_p_2_15","sub_p_2_19","sub_p_2_27","sub_p_2_31","sub_p_2_39","sub_p_2_43","sub_p_2_47"]

OUT_DIR = r"F:/tfr_single_trial_cycle_8"
EPOCHS_DIR = r"F:/epochs_ITC"

conds = {"int_rep": "cue/sat/int/rep", "int_swi": "cue/sat/int/swi", "ext_rep": "cue/sat/ext/rep","ext_swi": "cue/sat/ext/swi"}

bands = {"Low Beta": (13, 16),"Beta": (16, 28),"alpha": (8, 15),"theta": (4, 7)}

BIN = 0.25
N_PERM = 2000
ALPHA = 0.05
TAIL = 0
SEED = 42

def safe_cond_indices(epochs, cond):
    cond_sel = epochs[cond].selection
    idx = np.flatnonzero(np.isin(epochs.selection, cond_sel))
    return idx

def cond_mean(X, epochs, cond):
    idx = safe_cond_indices(epochs, cond)
    if len(idx) == 0:
        raise RuntimeError(f"No trials for {cond}")
    return X[idx].mean(axis=0)  # (ch, freq, time)

def band_average(M, freqs, f_lo, f_hi):
    fmask = (freqs >= f_lo) & (freqs <= f_hi)
    if not np.any(fmask):
        raise RuntimeError(f"No freq bins in [{f_lo}, {f_hi}]")
    return M[:, fmask, :].mean(axis=1)  # (ch, time)

def build_subject_effects(subj, freqs_ref=None, times_ref=None, ch_ref=None):
    power_path = os.path.join(OUT_DIR, f"{subj}_power.npy")
    meta_path  = os.path.join(OUT_DIR, f"{subj}_meta.npz")
    epo_path   = os.path.join(EPOCHS_DIR, f"{subj}-raw-ica-reject-ERP-epo.fif")

    if not (os.path.exists(power_path) and os.path.exists(meta_path) and os.path.exists(epo_path)):
        return None, None, None, f"missing file(s)"

    X = np.load(power_path, mmap_mode="r")  # (trial, ch, freq, time)
    meta = np.load(meta_path, allow_pickle=True)
    ch_names = meta["ch_names"].tolist()
    freqs = meta["freqs"]
    times = meta["times"]

    # axis consistency check
    if freqs_ref is not None:
        if len(freqs) != len(freqs_ref) or not np.allclose(freqs, freqs_ref):
            return None, None, None, "freq mismatch"
    if times_ref is not None:
        if len(times) != len(times_ref) or not np.allclose(times, times_ref):
            return None, None, None, "time mismatch"
    if ch_ref is not None:
        if ch_names != ch_ref:
            return None, None, None, "channel order mismatch"

    # epochs: only used for condition trial indexing
    epochs = mne.read_epochs(epo_path, preload=True).crop(tmin=times[0], tmax=times[-1])

    if X.shape[0] != len(epochs):
        return None, None, None, f"trial count mismatch: X={X.shape[0]} vs epochs={len(epochs)}"

    # condition means: (ch,freq,time)
    M = {k: cond_mean(X, epochs, c) for k, c in conds.items()}

    # effects in full TF (ch,freq,time)
    INT = 0.5 * (M["int_rep"] + M["int_swi"])
    EXT = 0.5 * (M["ext_rep"] + M["ext_swi"])
    D_int_ext = INT - EXT

    SWI = 0.5 * (M["int_swi"] + M["ext_swi"])
    REP = 0.5 * (M["int_rep"] + M["ext_rep"])
    D_swi_rep = SWI - REP

    D_inter = (M["int_swi"] - M["int_rep"]) - (M["ext_swi"] - M["ext_rep"])

    out = {
        "rule_switch_int_ext": D_int_ext,
        "att_switch_swi_rep": D_swi_rep,
        "interaction": D_inter,
    }
    return out, freqs, times, None

def make_time_bins(times, bin_s=0.25):
    t0, t1 = float(times[0]), float(times[-1])
    edges = np.arange(t0, t1 + 1e-9, bin_s)
    if edges[-1] < t1:
        edges = np.append(edges, t1)
    bins = []
    for a, b in zip(edges[:-1], edges[1:]):
        mask = (times >= a) & (times < b) if b < t1 else (times >= a) & (times <= b)
        if mask.any():
            bins.append((a, b, mask))
    return bins

def run_cluster_time_ch(X_subj_ch_time, info, adjacency, alpha=0.05, n_perm=2000, tail=0, seed=42):
    # MNE expects (n_samples, n_times, n_space) for spatio-temporal with adjacency over space
    X_st = np.transpose(X_subj_ch_time, (0, 2, 1))  # -> (subj, time, ch)

    T_obs, clusters, pvals, H0 = mne.stats.spatio_temporal_cluster_1samp_test(
        X_st,
        adjacency=adjacency,
        n_permutations=n_perm,
        threshold=None,   # uses t-threshold internally (like your logs showed ~2.01)
        tail=tail,
        out_type="mask",
        seed=seed,
        verbose=True,
    )

    # Build sig mask over (time,ch)
    sig = np.zeros(T_obs.shape, dtype=bool)  # (time,ch)
    for cl, p in zip(clusters, pvals):
        if p < alpha:
            sig |= cl  # cl is boolean mask if out_type="mask"

    # convert to (ch,time) for topomap usage
    sig_ch_time = sig.T  # (ch,time)
    return T_obs, clusters, pvals, sig_ch_time

# 1) Load one subject to anchor info/adjacency + axes
anchor = subjects[0]
epo_anchor = os.path.join(EPOCHS_DIR, f"{anchor}-raw-ica-reject-ERP-epo.fif")
epochs_anchor = mne.read_epochs(epo_anchor, preload=True)
info = epochs_anchor.copy().pick_types(eeg=True).info

adjacency, ch_names_adj = mne.channels.find_ch_adjacency(info, ch_type="eeg")
print("[adjacency] shape:", adjacency.shape)

# 2) Build subject stacks for each effect+band
effects = ["att_switch_swi_rep", "rule_switch_int_ext", "interaction"]

# store: band -> effect -> list of (ch,time)
stack = {band: {eff: [] for eff in effects} for band in bands.keys()}

freqs_ref = None
times_ref = None
ch_ref = None
kept = 0

for subj in subjects:
    out, freqs, times, err = build_subject_effects(subj, freqs_ref, times_ref, ch_ref)
    if err is not None:
        print("SKIP", subj, "->", err)
        continue

    # lock references on first kept subject
    if freqs_ref is None:
        freqs_ref = freqs.copy()
        times_ref = times.copy()
        # channel names from your meta (must match X channel order)
        meta0 = np.load(os.path.join(OUT_DIR, f"{subj}_meta.npz"), allow_pickle=True)
        ch_ref = meta0["ch_names"].tolist()

    # band-average and collect
    for band_name, (f_lo, f_hi) in bands.items():
        for eff in effects:
            D_ch_f_t = out[eff]  # (ch,freq,time)
            D_ch_t = band_average(D_ch_f_t, freqs_ref, f_lo, f_hi)  # (ch,time)
            stack[band_name][eff].append(D_ch_t.astype(np.float32))

    kept += 1

print("Kept subjects:", kept)

# 3) Cluster + plot: many topomaps per time-bin with black dots
time_bins = make_time_bins(times_ref, BIN)
print("n_time_bins:", len(time_bins), "bin_s:", BIN)

for band_name, (f_lo, f_hi) in bands.items():
    all_maps = []
    for eff in effects:
        X_eff = np.stack(stack[band_name][eff], axis=0)  
        all_maps.append(X_eff.mean(axis=0))           
    all_maps = np.concatenate(all_maps, axis=1)
    vmin, vmax = np.percentile(all_maps, [5, 95])

    n_rows = len(effects)
    n_cols = len(time_bins)

    fig = plt.figure(figsize=(18, 10))
    fig.suptitle(
        f"{band_name.upper()} ({f_lo:.0f}-{f_hi:.0f} Hz) — Cluster permutation topomaps",
        y=0.98
    )

    # +1 column reserved for colorbar
    gs = gridspec.GridSpec(
        n_rows,
        n_cols + 1,
        width_ratios=[1] * n_cols + [0.05],
        wspace=0.25,
        hspace=0.3
    )

    #plot
    for r, eff in enumerate(effects):
        X_eff = np.stack(stack[band_name][eff], axis=0)  # (subj,ch,time)

        T_obs, clusters, pvals, sig_ch_time = run_cluster_time_ch(
            X_eff, info, adjacency,
            alpha=ALPHA, n_perm=N_PERM, tail=TAIL, seed=SEED
        )

        mean_ch_t = X_eff.mean(axis=0)  # (ch,time)

        for c, (ta, tb, tmask) in enumerate(time_bins):
            ax = fig.add_subplot(gs[r, c])

            dat = mean_ch_t[:, tmask].mean(axis=1)
            sig_ch = sig_ch_time[:, tmask].any(axis=1)

            mne.viz.plot_topomap(
                dat,
                info,
                axes=ax,
                show=False,
                vlim=(vmin, vmax),
                contours=0,
                sensors=False,
                mask=sig_ch,
                mask_params=dict(
                    markersize=6,
                    markerfacecolor="k",
                    markeredgecolor="k"
                )
            )

            ax.set_title(f"{ta:+.2f}–{tb:+.2f}s", fontsize=8)
            if c == 0:
                ax.set_ylabel(eff, fontsize=10)

    #  colorbar (single, dedicated axis) 
    cax = fig.add_subplot(gs[:, -1])
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="RdBu_r")
    sm.set_array([])

    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label("Power (baseline-corrected logratio)", rotation=90)

    plt.show()

In [ ]:
import numpy as np
import mne
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import os
os.environ["QT_API"] = "pyqt5"
%matplotlib qt

subjects = ["sub_m_1_02", "sub_m_1_06", "sub_m_1_10","sub_m_1_14","sub_m_1_18","sub_m_1_22","sub_m_1_26","sub_m_1_30","sub_m_1_34","sub_m_1_38","sub_m_1_42","sub_m_1_46",
            "sub_m_2_04","sub_m_2_08","sub_m_2_12","sub_m_2_16","sub_m_2_20","sub_m_2_24","sub_m_2_28","sub_m_2_32","sub_m_2_36","sub_m_2_40","sub_m_2_44",
            "sub_p_1_01","sub_p_1_05","sub_p_1_09","sub_p_1_13","sub_p_1_17","sub_p_1_21","sub_p_1_25","sub_p_1_29","sub_p_1_33","sub_p_1_35","sub_p_1_37","sub_p_1_41","sub_p_1_45",
            "sub_p_2_03","sub_p_2_07","sub_p_2_11","sub_p_2_15","sub_p_2_19","sub_p_2_27","sub_p_2_31","sub_p_2_39","sub_p_2_43","sub_p_2_47"]

OUT_DIR = r"F:/tfr_single_trial_cycle_8"
EPOCHS_DIR = r"F:/epochs_ITC"

conds = {"int_rep": "cue/cue2/int/rep", "int_swi": "cue/cue2/int/swi", "ext_rep": "cue/cue2/ext/rep","ext_swi": "cue/cue2/ext/swi"}

bands = {"Low Beta": (13, 16),"Beta": (16, 28),"alpha": (8, 15),"theta": (4, 7)}

BIN = 0.25
N_PERM = 1000
ALPHA = 0.05
TAIL = 0
SEED = 42

ANALYSIS_TMIN = 0.3
ANALYSIS_TMAX = 1.0

def safe_cond_indices(epochs, cond):
    cond_sel = epochs[cond].selection
    idx = np.flatnonzero(np.isin(epochs.selection, cond_sel))
    return idx

def cond_mean(X, epochs, cond):
    idx = safe_cond_indices(epochs, cond)
    if len(idx) == 0:
        raise RuntimeError(f"No trials for {cond}")
    return X[idx].mean(axis=0)  # (ch, freq, time)

def band_average(M, freqs, f_lo, f_hi):
    fmask = (freqs >= f_lo) & (freqs <= f_hi)
    if not np.any(fmask):
        raise RuntimeError(f"No freq bins in [{f_lo}, {f_hi}]")
    return M[:, fmask, :].mean(axis=1)  # (ch, time)

def build_subject_effects(subj, freqs_ref=None, times_ref=None, ch_ref=None):
    power_path = os.path.join(OUT_DIR, f"{subj}_power.npy")
    meta_path  = os.path.join(OUT_DIR, f"{subj}_meta.npz")
    epo_path   = os.path.join(EPOCHS_DIR, f"{subj}-raw-ica-reject-ERP-epo.fif")

    if not (os.path.exists(power_path) and os.path.exists(meta_path) and os.path.exists(epo_path)):
        return None, None, None, f"missing file(s)"

    X = np.load(power_path, mmap_mode="r")  # (trial, ch, freq, time)
    meta = np.load(meta_path, allow_pickle=True)
    ch_names = meta["ch_names"].tolist()
    freqs = meta["freqs"]
    times = meta["times"]

    # axis consistency check
    if freqs_ref is not None:
        if len(freqs) != len(freqs_ref) or not np.allclose(freqs, freqs_ref):
            return None, None, None, "freq mismatch"
    if times_ref is not None:
        if len(times) != len(times_ref) or not np.allclose(times, times_ref):
            return None, None, None, "time mismatch"
    if ch_ref is not None:
        if ch_names != ch_ref:
            return None, None, None, "channel order mismatch"

    # epochs: only used for condition trial indexing
    epochs = mne.read_epochs(epo_path, preload=True).crop(tmin=times[0], tmax=times[-1])

    if X.shape[0] != len(epochs):
        return None, None, None, f"trial count mismatch: X={X.shape[0]} vs epochs={len(epochs)}"

    # condition means: (ch,freq,time)
    M = {k: cond_mean(X, epochs, c) for k, c in conds.items()}

    # effects in full TF (ch,freq,time)
    INT = 0.5 * (M["int_rep"] + M["int_swi"])
    EXT = 0.5 * (M["ext_rep"] + M["ext_swi"])
    D_int_ext = INT - EXT

    SWI = 0.5 * (M["int_swi"] + M["ext_swi"])
    REP = 0.5 * (M["int_rep"] + M["ext_rep"])
    D_swi_rep = SWI - REP

    D_inter = (M["int_swi"] - M["int_rep"]) - (M["ext_swi"] - M["ext_rep"])

    out = {
        "rule_switch_int_ext": D_int_ext,
        "att_switch_swi_rep": D_swi_rep,
        "interaction": D_inter,
    }
    return out, freqs, times, None

def make_time_bins(times, bin_s=0.25):
    t0, t1 = float(times[0]), float(times[-1])
    edges = np.arange(t0, t1 + 1e-9, bin_s)
    if edges[-1] < t1:
        edges = np.append(edges, t1)
    bins = []
    for a, b in zip(edges[:-1], edges[1:]):
        mask = (times >= a) & (times < b) if b < t1 else (times >= a) & (times <= b)
        if mask.any():
            bins.append((a, b, mask))
    return bins

def run_cluster_time_ch(X_subj_ch_time, info, adjacency, alpha=0.05, n_perm=2000, tail=0, seed=42):
    # MNE expects (n_samples, n_times, n_space) for spatio-temporal with adjacency over space
    X_st = np.transpose(X_subj_ch_time, (0, 2, 1))  # -> (subj, time, ch)

    T_obs, clusters, pvals, H0 = mne.stats.spatio_temporal_cluster_1samp_test(
        X_st,
        adjacency=adjacency,
        n_permutations=n_perm,
        threshold=None,   # uses t-threshold internally (like your logs showed ~2.01)
        tail=tail,
        out_type="mask",
        seed=seed,
        verbose=True,
    )

    # Build sig mask over (time,ch)
    sig = np.zeros(T_obs.shape, dtype=bool)  # (time,ch)
    for cl, p in zip(clusters, pvals):
        if p < alpha:
            sig |= cl  # cl is boolean mask if out_type="mask"

    # convert to (ch,time) for topomap usage
    sig_ch_time = sig.T  # (ch,time)
    return T_obs, clusters, pvals, sig_ch_time

# 1) Load one subject to anchor info/adjacency + axes
anchor = subjects[0]
epo_anchor = os.path.join(EPOCHS_DIR, f"{anchor}-raw-ica-reject-ERP-epo.fif")
epochs_anchor = mne.read_epochs(epo_anchor, preload=True)
info = epochs_anchor.copy().pick_types(eeg=True).info

adjacency, ch_names_adj = mne.channels.find_ch_adjacency(info, ch_type="eeg")
print("[adjacency] shape:", adjacency.shape)

# 2) Build subject stacks for each effect+band
effects = ["att_switch_swi_rep", "rule_switch_int_ext", "interaction"]

# store: band -> effect -> list of (ch,time)
stack = {band: {eff: [] for eff in effects} for band in bands.keys()}

freqs_ref = None
times_ref = None
ch_ref = None
kept = 0

for subj in subjects:
    out, freqs, times, err = build_subject_effects(subj, freqs_ref, times_ref, ch_ref)
    if err is not None:
        print("SKIP", subj, "->", err)
        continue

    # lock references on first kept subject
    if freqs_ref is None:
        freqs_ref = freqs.copy()
        times_ref = times.copy()
        # channel names from your meta (must match X channel order)
        meta0 = np.load(os.path.join(OUT_DIR, f"{subj}_meta.npz"), allow_pickle=True)
        ch_ref = meta0["ch_names"].tolist()

    # band-average and collect
    for band_name, (f_lo, f_hi) in bands.items():
        for eff in effects:
            D_ch_f_t = out[eff]  # (ch,freq,time)
            D_ch_t = band_average(D_ch_f_t, freqs_ref, f_lo, f_hi)  # (ch,time)
            stack[band_name][eff].append(D_ch_t.astype(np.float32))

    kept += 1

print("Kept subjects:", kept)

# 3) Cluster + plot: many topomaps per time-bin with black dots
time_bins = make_time_bins(times_ref, BIN)
print("n_time_bins:", len(time_bins), "bin_s:", BIN)

for band_name, (f_lo, f_hi) in bands.items():
    all_maps = []
    for eff in effects:
        X_eff = np.stack(stack[band_name][eff], axis=0)  
        all_maps.append(X_eff.mean(axis=0))           
    all_maps = np.concatenate(all_maps, axis=1)
    vmin, vmax = np.percentile(all_maps, [5, 95])

    n_rows = len(effects)
    n_cols = len(time_bins)

    fig = plt.figure(figsize=(18, 10))
    fig.suptitle(
        f"{band_name.upper()} ({f_lo:.0f}-{f_hi:.0f} Hz) — Cluster permutation topomaps",
        y=0.98
    )

    # +1 column reserved for colorbar
    gs = gridspec.GridSpec(
        n_rows,
        n_cols + 1,
        width_ratios=[1] * n_cols + [0.05],
        wspace=0.25,
        hspace=0.3
    )

    #plot
    for r, eff in enumerate(effects):
        X_eff = np.stack(stack[band_name][eff], axis=0)  # (subj,ch,time)

        T_obs, clusters, pvals, sig_ch_time = run_cluster_time_ch(
            X_eff, info, adjacency,
            alpha=ALPHA, n_perm=N_PERM, tail=TAIL, seed=SEED
        )

        mean_ch_t = X_eff.mean(axis=0)  # (ch,time)

        for c, (ta, tb, tmask) in enumerate(time_bins):
            ax = fig.add_subplot(gs[r, c])

            dat = mean_ch_t[:, tmask].mean(axis=1)
            sig_ch = sig_ch_time[:, tmask].any(axis=1)

            mne.viz.plot_topomap(
                dat,
                info,
                axes=ax,
                show=False,
                vlim=(vmin, vmax),
                contours=0,
                sensors=False,
                mask=sig_ch,
                mask_params=dict(
                    markersize=6,
                    markerfacecolor="k",
                    markeredgecolor="k"
                )
            )

            ax.set_title(f"{ta:+.2f}–{tb:+.2f}s", fontsize=8)
            if c == 0:
                ax.set_ylabel(eff, fontsize=10)

    #  colorbar (single, dedicated axis) 
    cax = fig.add_subplot(gs[:, -1])
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="RdBu_r")
    sm.set_array([])

    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label("Power (baseline-corrected logratio)", rotation=90)

    plt.show()

In [ ]:
import numpy as np
import mne
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import os
os.environ["QT_API"] = "pyqt5"
%matplotlib qt

subjects = ["sub_m_1_02", "sub_m_1_06", "sub_m_1_10","sub_m_1_14","sub_m_1_18","sub_m_1_22","sub_m_1_26","sub_m_1_30","sub_m_1_34","sub_m_1_38","sub_m_1_42","sub_m_1_46",
            "sub_m_2_04","sub_m_2_08","sub_m_2_12","sub_m_2_16","sub_m_2_20","sub_m_2_24","sub_m_2_28","sub_m_2_32","sub_m_2_36","sub_m_2_40","sub_m_2_44",
            "sub_p_1_01","sub_p_1_05","sub_p_1_09","sub_p_1_13","sub_p_1_17","sub_p_1_21","sub_p_1_25","sub_p_1_29","sub_p_1_33","sub_p_1_35","sub_p_1_37","sub_p_1_41","sub_p_1_45",
            "sub_p_2_03","sub_p_2_07","sub_p_2_11","sub_p_2_15","sub_p_2_19","sub_p_2_27","sub_p_2_31","sub_p_2_39","sub_p_2_43","sub_p_2_47"]

POWER_FNAME = "{subj}_power.npy"   
META_FNAME  = "{subj}_meta.npz"    
OUT_DIR = r"F:/tfr_single_trial_cycle_8"
EPOCHS_DIR = r"F:/epochs_ITC"
EPO_FNAME  = "{subj}-raw-ica-reject-ERP-epo.fif" 

conds = {"int_rep": "cue/cue2/int/rep", "int_swi": "cue/cue2/int/swi", "ext_rep": "cue/cue2/ext/rep","ext_swi": "cue/cue2/ext/swi"}

bands = {"Low Beta": (13, 16),"Beta": (16, 28),"alpha": (8, 15),"theta": (4, 7)}

BIN = 0.25
N_PERM = 1000
ALPHA = 0.05
TAIL = 0
SEED = 42

ANALYSIS_TMIN = 0.3
ANALYSIS_TMAX = 1.0


def safe_cond_indices(epochs: mne.Epochs, cond: str) -> np.ndarray:
    cond_sel = epochs[cond].selection          # indices in raw (original) space
    idx = np.flatnonzero(np.isin(epochs.selection, cond_sel))  # indices into current epochs object
    return idx

def cond_mean(X: np.ndarray, epochs: mne.Epochs, cond: str) -> np.ndarray:
    idx = safe_cond_indices(epochs, cond)
    if len(idx) == 0:
        raise RuntimeError(f"No trials for {cond}")
    return X[idx].mean(axis=0)

def band_average(M_ch_f_t: np.ndarray, freqs: np.ndarray, f_lo: float, f_hi: float) -> np.ndarray:
    fmask = (freqs >= f_lo) & (freqs <= f_hi)
    if not np.any(fmask):
        raise RuntimeError(f"No freq bins in [{f_lo}, {f_hi}]")
    return M_ch_f_t[:, fmask, :].mean(axis=1)

def make_time_bins(times: np.ndarray, bin_s: float = 0.25):
    t0, t1 = float(times[0]), float(times[-1])
    edges = np.arange(t0, t1 + 1e-9, bin_s)
    if edges[-1] < t1:
        edges = np.append(edges, t1)
    bins = []
    for a, b in zip(edges[:-1], edges[1:]):
        mask = (times >= a) & (times < b) if b < t1 else (times >= a) & (times <= b)
        if mask.any():
            bins.append((a, b, mask))
    return bins

def run_cluster_time_ch(X_subj_ch_time: np.ndarray, adjacency, alpha=0.05, n_perm=2000, tail=0, seed=42):
    X_st = np.transpose(X_subj_ch_time, (0, 2, 1))  # -> (subj, time, ch)

    T_obs, clusters, pvals, H0 = mne.stats.spatio_temporal_cluster_1samp_test(
        X_st,
        adjacency=adjacency,
        n_permutations=n_perm,
        threshold=None,
        tail=tail,
        out_type="mask",
        seed=seed,
        verbose=True,
    )

    sig = np.zeros(T_obs.shape, dtype=bool)  # (time,ch)
    for cl, p in zip(clusters, pvals):
        if p < alpha:
            sig |= cl

    sig_ch_time = sig.T  # (ch,time)
    return T_obs, clusters, pvals, sig_ch_time

def build_subject_effects(subj: str, freqs_ref=None, times_ref=None, ch_ref=None):

    power_path = os.path.join(OUT_DIR, POWER_FNAME.format(subj=subj))
    meta_path  = os.path.join(OUT_DIR, META_FNAME.format(subj=subj))
    epo_path   = os.path.join(EPOCHS_DIR, EPO_FNAME.format(subj=subj))

    if not (os.path.exists(power_path) and os.path.exists(meta_path) and os.path.exists(epo_path)):
        return None, None, None, None, "missing file(s)"

    X = np.load(power_path, mmap_mode="r")  # (trial, ch, freq, time)
    meta = np.load(meta_path, allow_pickle=True)
    ch_names = meta["ch_names"].tolist()
    freqs = meta["freqs"].astype(float)
    times = meta["times"].astype(float)

    # consistency checks across subjects
    if freqs_ref is not None and (len(freqs) != len(freqs_ref) or not np.allclose(freqs, freqs_ref)):
        return None, None, None, None, "freq mismatch"
    if times_ref is not None and (len(times) != len(times_ref) or not np.allclose(times, times_ref)):
        return None, None, None, None, "time mismatch"
    if ch_ref is not None and (ch_names != ch_ref):
        return None, None, None, None, "channel order mismatch"

    # Epochs are for trial indexing only; crop to match TFR time axis (harmless if already aligned)
    epochs = mne.read_epochs(epo_path, preload=True).crop(tmin=times[0], tmax=times[-1])

    if X.shape[0] != len(epochs):
        return None, None, None, None, f"trial count mismatch: X={X.shape[0]} vs epochs={len(epochs)}"

    # condition means: (ch,freq,time)
    M = {k: cond_mean(X, epochs, c) for k, c in conds.items()}

    # effects in full TF (ch,freq,time)
    INT = 0.5 * (M["int_rep"] + M["int_swi"])
    EXT = 0.5 * (M["ext_rep"] + M["ext_swi"])
    D_int_ext = INT - EXT

    SWI = 0.5 * (M["int_swi"] + M["ext_swi"])
    REP = 0.5 * (M["int_rep"] + M["ext_rep"])
    D_swi_rep = SWI - REP

    D_inter = (M["int_swi"] - M["int_rep"]) - (M["ext_swi"] - M["ext_rep"])

    out = {
        "rule_switch_int_ext": D_int_ext,
        "att_switch_swi_rep": D_swi_rep,
        "interaction": D_inter,
    }
    return out, freqs, times, ch_names, None


# 2) ANCHOR: adjacency + info
anchor = subjects[0]
epo_anchor = os.path.join(EPOCHS_DIR, EPO_FNAME.format(subj=anchor))
epochs_anchor = mne.read_epochs(epo_anchor, preload=True)
info = epochs_anchor.copy().pick_types(eeg=True, eog=False, ecg=False, stim=False, misc=False).info

adjacency, ch_names_adj = mne.channels.find_ch_adjacency(info, ch_type="eeg")
print("[adjacency] shape:", adjacency.shape)

effects = ["att_switch_swi_rep", "rule_switch_int_ext", "interaction"]
stack = {band: {eff: [] for eff in effects} for band in bands.keys()}

freqs_ref = None
times_ref = None
ch_ref = None

kept = 0
skipped = {}

# We'll create the analysis time mask after we lock times_ref (on first kept subject)
tmask_analysis = None


# 3) BUILD STACKS (STRICTLY 0.3–1.0 s ONLY)
for subj in subjects:
    out, freqs, times, ch_names, err = build_subject_effects(subj, freqs_ref, times_ref, ch_ref)
    if err is not None:
        skipped[subj] = err
        print("SKIP", subj, "->", err)
        continue

    # lock references on first kept subject
    if freqs_ref is None:
        freqs_ref = freqs.copy()
        times_ref = times.copy()
        ch_ref = ch_names[:]  # channel order from meta must match X

        # build analysis time mask ONCE
        tmask_analysis = (times_ref >= ANALYSIS_TMIN) & (times_ref <= ANALYSIS_TMAX)
        if not np.any(tmask_analysis):
            raise RuntimeError(
                f"No time points in analysis window {ANALYSIS_TMIN}..{ANALYSIS_TMAX}. "
                f"Available times: {times_ref[0]}..{times_ref[-1]}"
            )
        # restrict times_ref to analysis window for downstream bins/titles
        times_ref = times_ref[tmask_analysis]

        print(f"[analysis] Restricting to {ANALYSIS_TMIN}..{ANALYSIS_TMAX} s -> n_times={len(times_ref)}")

    # band-average and collect (after time restriction!)
    for band_name, (f_lo, f_hi) in bands.items():
        for eff in effects:
            D_ch_f_t = out[eff]                              # (ch,freq,time) full time
            D_ch_t_full = band_average(D_ch_f_t, freqs_ref, f_lo, f_hi)  # (ch,time) full time
            D_ch_t = D_ch_t_full[:, tmask_analysis]          # (ch,time) ONLY analysis window
            stack[band_name][eff].append(D_ch_t.astype(np.float32))

    kept += 1

print("Kept subjects:", kept, "/", len(subjects))
if skipped:
    print("Skipped subjects:", len(skipped))
    # print(skipped)  # uncomment to see all reasons


# 4) CLUSTER + PLOT (STRICTLY 0.3–1.0 s)
time_bins = make_time_bins(times_ref, BIN)
print("n_time_bins:", len(time_bins), "bin_s:", BIN)

for band_name, (f_lo, f_hi) in bands.items():
    # set color range across all effects for this band (robust via percentiles)
    all_maps = []
    for eff in effects:
        X_eff = np.stack(stack[band_name][eff], axis=0)  # (subj,ch,time)
        all_maps.append(X_eff.mean(axis=0))              # (ch,time)
    all_maps = np.concatenate(all_maps, axis=1)          # (ch, time*effects)
    vmin, vmax = np.percentile(all_maps, [5, 95])

    n_rows = len(effects)
    n_cols = len(time_bins)

    fig = plt.figure(figsize=(18, 10))
    fig.suptitle(
        f"{band_name.upper()} ({f_lo:.0f}-{f_hi:.0f} Hz) — Cluster permutation topomaps — {ANALYSIS_TMIN:.1f}–{ANALYSIS_TMAX:.1f}s",
        y=0.98
    )

    gs = gridspec.GridSpec(
        n_rows,
        n_cols + 1,
        width_ratios=[1] * n_cols + [0.05],
        wspace=0.25,
        hspace=0.3
    )

    for r, eff in enumerate(effects):
        X_eff = np.stack(stack[band_name][eff], axis=0)  # (subj,ch,time) already restricted

        T_obs, clusters, pvals, sig_ch_time = run_cluster_time_ch(
            X_eff, adjacency,
            alpha=ALPHA, n_perm=N_PERM, tail=TAIL, seed=SEED
        )

        mean_ch_t = X_eff.mean(axis=0)  # (ch,time)

        for c, (ta, tb, tmask) in enumerate(time_bins):
            ax = fig.add_subplot(gs[r, c])

            dat = mean_ch_t[:, tmask].mean(axis=1)
            sig_ch = sig_ch_time[:, tmask].any(axis=1)

            mne.viz.plot_topomap(
                dat,
                info,
                axes=ax,
                show=False,
                vlim=(vmin, vmax),
                contours=0,
                sensors=False,
                mask=sig_ch,
                mask_params=dict(
                    markersize=6,
                    markerfacecolor="k",
                    markeredgecolor="k"
                )
            )

            ax.set_title(f"{ta:+.2f}–{tb:+.2f}s", fontsize=8)
            if c == 0:
                ax.set_ylabel(eff, fontsize=10)

    # colorbar axis
    cax = fig.add_subplot(gs[:, -1])
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="RdBu_r")
    sm.set_array([])

    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label("Power (baseline-corrected logratio)", rotation=90)

    plt.show()

In [ ]:
print("T_obs max abs:", np.max(np.abs(T_obs)))


In [ ]:
import numpy as np
import mne
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import os
os.environ["QT_API"] = "pyqt5"
%matplotlib qt

subjects = ["sub_m_1_02", "sub_m_1_06", "sub_m_1_10","sub_m_1_14","sub_m_1_18","sub_m_1_22","sub_m_1_26","sub_m_1_30","sub_m_1_34","sub_m_1_38","sub_m_1_42","sub_m_1_46",
            "sub_m_2_04","sub_m_2_08","sub_m_2_12","sub_m_2_16","sub_m_2_20","sub_m_2_24","sub_m_2_28","sub_m_2_32","sub_m_2_36","sub_m_2_40","sub_m_2_44",
            "sub_p_1_01","sub_p_1_05","sub_p_1_09","sub_p_1_13","sub_p_1_17","sub_p_1_21","sub_p_1_25","sub_p_1_29","sub_p_1_33","sub_p_1_35","sub_p_1_37","sub_p_1_41","sub_p_1_45",
            "sub_p_2_03","sub_p_2_07","sub_p_2_11","sub_p_2_15","sub_p_2_19","sub_p_2_27","sub_p_2_31","sub_p_2_39","sub_p_2_43","sub_p_2_47"]

OUT_DIR = r"F:/tfr_single_trial_cycle_8"
EPOCHS_DIR = r"F:/epochs_ITC"

conds = {"int_rep": "cue/int/rep/sat", "int_swi": "cue/int/swi/sat", "ext_rep": "cue/ext/rep/sat","ext_swi": "cue/ext/swi/sat"}

bands = {"Low Beta": (13, 16),"Beta": (16, 28),"alpha": (8, 15),"theta": (4, 7)}

BIN = 0.25
N_PERM = 1000
ALPHA = 0.05
TAIL = 0
SEED = 42

def safe_cond_indices(epochs, cond):
    cond_sel = epochs[cond].selection
    idx = np.flatnonzero(np.isin(epochs.selection, cond_sel))
    return idx

def cond_mean(X, epochs, cond):
    idx = safe_cond_indices(epochs, cond)
    if len(idx) == 0:
        raise RuntimeError(f"No trials for {cond}")
    return X[idx].mean(axis=0)  # (ch, freq, time)

def band_average(M, freqs, f_lo, f_hi):
    fmask = (freqs >= f_lo) & (freqs <= f_hi)
    if not np.any(fmask):
        raise RuntimeError(f"No freq bins in [{f_lo}, {f_hi}]")
    return M[:, fmask, :].mean(axis=1)  # (ch, time)

def build_subject_effects(subj, freqs_ref=None, times_ref=None, ch_ref=None):
    power_path = os.path.join(OUT_DIR, f"{subj}_power.npy")
    meta_path  = os.path.join(OUT_DIR, f"{subj}_meta.npz")
    epo_path   = os.path.join(EPOCHS_DIR, f"{subj}-raw-ica-reject-ERP-epo.fif")

    if not (os.path.exists(power_path) and os.path.exists(meta_path) and os.path.exists(epo_path)):
        return None, None, None, f"missing file(s)"

    X = np.load(power_path, mmap_mode="r")  # (trial, ch, freq, time)
    meta = np.load(meta_path, allow_pickle=True)
    ch_names = meta["ch_names"].tolist()
    freqs = meta["freqs"]
    times = meta["times"]

    # axis consistency check
    if freqs_ref is not None:
        if len(freqs) != len(freqs_ref) or not np.allclose(freqs, freqs_ref):
            return None, None, None, "freq mismatch"
    if times_ref is not None:
        if len(times) != len(times_ref) or not np.allclose(times, times_ref):
            return None, None, None, "time mismatch"
    if ch_ref is not None:
        if ch_names != ch_ref:
            return None, None, None, "channel order mismatch"

    # epochs: only used for condition trial indexing
    epochs = mne.read_epochs(epo_path, preload=True).crop(tmin=times[0], tmax=times[-1])

    if X.shape[0] != len(epochs):
        return None, None, None, f"trial count mismatch: X={X.shape[0]} vs epochs={len(epochs)}"

    # condition means: (ch,freq,time)
    M = {k: cond_mean(X, epochs, c) for k, c in conds.items()}

    # effects in full TF (ch,freq,time)
    INT = 0.5 * (M["int_rep"] + M["int_swi"])
    EXT = 0.5 * (M["ext_rep"] + M["ext_swi"])
    D_int_ext = INT - EXT

    SWI = 0.5 * (M["int_swi"] + M["ext_swi"])
    REP = 0.5 * (M["int_rep"] + M["ext_rep"])
    D_swi_rep = SWI - REP

    D_inter = (M["int_swi"] - M["int_rep"]) - (M["ext_swi"] - M["ext_rep"])

    out = {
        "rule_switch_int_ext": D_int_ext,
        "att_switch_swi_rep": D_swi_rep,
        "interaction": D_inter,
    }
    return out, freqs, times, None

def make_time_bins(times, bin_s=0.25):
    t0, t1 = float(times[0]), float(times[-1])
    edges = np.arange(t0, t1 + 1e-9, bin_s)
    if edges[-1] < t1:
        edges = np.append(edges, t1)
    bins = []
    for a, b in zip(edges[:-1], edges[1:]):
        mask = (times >= a) & (times < b) if b < t1 else (times >= a) & (times <= b)
        if mask.any():
            bins.append((a, b, mask))
    return bins

def run_cluster_time_ch(X_subj_ch_time, info, adjacency, alpha=0.05, n_perm=2000, tail=0, seed=42):
    # MNE expects (n_samples, n_times, n_space) for spatio-temporal with adjacency over space
    X_st = np.transpose(X_subj_ch_time, (0, 2, 1))  # -> (subj, time, ch)

    T_obs, clusters, pvals, H0 = mne.stats.spatio_temporal_cluster_1samp_test(
        X_st,
        adjacency=adjacency,
        n_permutations=n_perm,
        threshold=None,   # uses t-threshold internally (like your logs showed ~2.01)
        tail=tail,
        out_type="mask",
        seed=seed,
        verbose=True,
    )

    # Build sig mask over (time,ch)
    sig = np.zeros(T_obs.shape, dtype=bool)  # (time,ch)
    for cl, p in zip(clusters, pvals):
        if p < alpha:
            sig |= cl  # cl is boolean mask if out_type="mask"

    # convert to (ch,time) for topomap usage
    sig_ch_time = sig.T  # (ch,time)
    return T_obs, clusters, pvals, sig_ch_time

# 1) Load one subject to anchor info/adjacency + axes
anchor = subjects[0]
epo_anchor = os.path.join(EPOCHS_DIR, f"{anchor}-raw-ica-reject-ERP-epo.fif")
epochs_anchor = mne.read_epochs(epo_anchor, preload=True)
info = epochs_anchor.copy().pick_types(eeg=True).info

adjacency, ch_names_adj = mne.channels.find_ch_adjacency(info, ch_type="eeg")
print("[adjacency] shape:", adjacency.shape)

# 2) Build subject stacks for each effect+band
effects = ["att_switch_swi_rep", "rule_switch_int_ext", "interaction"]

# store: band -> effect -> list of (ch,time)
stack = {band: {eff: [] for eff in effects} for band in bands.keys()}

freqs_ref = None
times_ref = None
ch_ref = None
kept = 0

for subj in subjects:
    out, freqs, times, err = build_subject_effects(subj, freqs_ref, times_ref, ch_ref)
    if err is not None:
        print("SKIP", subj, "->", err)
        continue

    # lock references on first kept subject
    if freqs_ref is None:
        freqs_ref = freqs.copy()
        times_ref = times.copy()
        # channel names from your meta (must match X channel order)
        meta0 = np.load(os.path.join(OUT_DIR, f"{subj}_meta.npz"), allow_pickle=True)
        ch_ref = meta0["ch_names"].tolist()

    # band-average and collect
    for band_name, (f_lo, f_hi) in bands.items():
        for eff in effects:
            D_ch_f_t = out[eff]  # (ch,freq,time)
            D_ch_t = band_average(D_ch_f_t, freqs_ref, f_lo, f_hi)  # (ch,time)
            stack[band_name][eff].append(D_ch_t.astype(np.float32))

    kept += 1

print("Kept subjects:", kept)

# 3) Cluster + plot: many topomaps per time-bin with black dots
time_bins = make_time_bins(times_ref, BIN)
print("n_time_bins:", len(time_bins), "bin_s:", BIN)

for band_name, (f_lo, f_hi) in bands.items():
    all_maps = []
    for eff in effects:
        X_eff = np.stack(stack[band_name][eff], axis=0)  
        all_maps.append(X_eff.mean(axis=0))           
    all_maps = np.concatenate(all_maps, axis=1)
    vmin, vmax = np.percentile(all_maps, [5, 95])

    n_rows = len(effects)
    n_cols = len(time_bins)

    fig = plt.figure(figsize=(18, 10))
    fig.suptitle(
        f"{band_name.upper()} ({f_lo:.0f}-{f_hi:.0f} Hz) — Cluster permutation topomaps",
        y=0.98
    )

    # +1 column reserved for colorbar
    gs = gridspec.GridSpec(
        n_rows,
        n_cols + 1,
        width_ratios=[1] * n_cols + [0.05],
        wspace=0.25,
        hspace=0.3
    )

    #plot
    for r, eff in enumerate(effects):
        X_eff = np.stack(stack[band_name][eff], axis=0)  # (subj,ch,time)

        T_obs, clusters, pvals, sig_ch_time = run_cluster_time_ch(
            X_eff, info, adjacency,
            alpha=ALPHA, n_perm=N_PERM, tail=TAIL, seed=SEED
        )

        mean_ch_t = X_eff.mean(axis=0)  # (ch,time)

        for c, (ta, tb, tmask) in enumerate(time_bins):
            ax = fig.add_subplot(gs[r, c])

            dat = mean_ch_t[:, tmask].mean(axis=1)
            sig_ch = sig_ch_time[:, tmask].any(axis=1)

            mne.viz.plot_topomap(
                dat,
                info,
                axes=ax,
                show=False,
                vlim=(vmin, vmax),
                contours=0,
                sensors=False,
                mask=sig_ch,
                mask_params=dict(
                    markersize=6,
                    markerfacecolor="k",
                    markeredgecolor="k"
                )
            )

            ax.set_title(f"{ta:+.2f}–{tb:+.2f}s", fontsize=8)
            if c == 0:
                ax.set_ylabel(eff, fontsize=10)

    #  colorbar (single, dedicated axis) 
    cax = fig.add_subplot(gs[:, -1])
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="RdBu_r")
    sm.set_array([])

    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label("Power (baseline-corrected logratio)", rotation=90)

    plt.show()

In [ ]:
#Checking data structure
import numpy as np
meta = np.load(r"F:\tfr_single_trial_cycle_8\sub_m_1_02_meta.npz")
meta.files
power = np.load(r"F:\tfr_single_trial_cycle_8\sub_m_1_02_power.npy")
power.shape

In [ ]:
import numpy as np
import mne
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import os
os.environ["QT_API"] = "pyqt5"
%matplotlib qt

subjects = ["sub_m_1_02", "sub_m_1_06", "sub_m_1_10","sub_m_1_14","sub_m_1_18","sub_m_1_22","sub_m_1_26","sub_m_1_30","sub_m_1_34","sub_m_1_38","sub_m_1_42","sub_m_1_46",
            "sub_m_2_04","sub_m_2_08","sub_m_2_12","sub_m_2_16","sub_m_2_20","sub_m_2_24","sub_m_2_28","sub_m_2_32","sub_m_2_36","sub_m_2_40","sub_m_2_44",
            "sub_p_1_01","sub_p_1_05","sub_p_1_09","sub_p_1_13","sub_p_1_17","sub_p_1_21","sub_p_1_25","sub_p_1_29","sub_p_1_33","sub_p_1_35","sub_p_1_37","sub_p_1_41","sub_p_1_45",
            "sub_p_2_03","sub_p_2_07","sub_p_2_11","sub_p_2_15","sub_p_2_19","sub_p_2_27","sub_p_2_31","sub_p_2_39","sub_p_2_43","sub_p_2_47"]

TFR_DIR = r"F:/tfr_single_trial_cycle_8"
EPO_DIR = r"F:/epochs_ITC"

META_FNAME  = "{sub}_meta.npz"
POWER_FNAME = "{sub}_power.npy"
EPO_FNAME = "{sub}-raw-ica-reject-ERP-epo.fif"

conds = {"int_rep": "targ/sat/int/rep", "int_swi": "targ/sat/int/swi", "ext_rep": "targ/sat/ext/rep","ext_swi": "targ/sat/ext/swi"}
bands = {"Low Beta": (13, 16),"Beta": (16, 28),"alpha": (8, 15),"theta": (4, 7)}

TMIN, TMAX = -0.2, 1.0

def safe_cond_indices(epochs: mne.Epochs, cond_key: str):
    """
    Return indices into X (power.npy) that correspond to epochs[cond_key].
    Works even if epochs has been dropped etc. because we map via epochs.selection.
    """
    sel = epochs[cond_key].selection
    idx = np.flatnonzero(np.isin(epochs.selection, sel))
    return idx

def load_subject_data(sub):
    power_path = os.path.join(TFR_DIR, POWER_FNAME.format(sub=sub))
    meta_path  = os.path.join(TFR_DIR, META_FNAME.format(sub=sub))
    epo_path   = os.path.join(EPO_DIR,  EPO_FNAME.format(sub=sub))

    if not os.path.exists(power_path):
        raise FileNotFoundError(power_path)
    if not os.path.exists(meta_path):
        raise FileNotFoundError(meta_path)
    if not os.path.exists(epo_path):
        raise FileNotFoundError(epo_path)

    X = np.load(power_path, mmap_mode="r")  # (trial, ch, freq, time)
    meta = np.load(meta_path, allow_pickle=True)

    ch_names = meta["ch_names"].tolist() if hasattr(meta["ch_names"], "tolist") else list(meta["ch_names"])
    freqs = meta["freqs"].astype(float)
    times = meta["times"].astype(float)

    epochs = mne.read_epochs(epo_path, preload=False, verbose=False)

    # align epoch time to TFR time if needed (for safety)
    # Only needed for plotting mask; trial indexing is independent of time axis.
    return X, ch_names, freqs, times, epochs

def band_average_ch_time(M_ch_f_t, freqs, f_lo, f_hi):
    fmask = (freqs >= f_lo) & (freqs <= f_hi)
    if not np.any(fmask):
        raise RuntimeError(f"No freq bins inside [{f_lo},{f_hi}] Hz")
    return M_ch_f_t[:, fmask, :].mean(axis=1)  # (ch,time)

def plot_heatmap_ch_time(data_ch_time, times, ch_names, title, vlim=None):
    plt.figure(figsize=(10, 6))
    if vlim is None:
        vmin, vmax = np.nanpercentile(data_ch_time, [5, 95])
    else:
        vmin, vmax = vlim

    plt.imshow(
        data_ch_time,
        aspect="auto",
        origin="lower",
        extent=[times[0], times[-1], 0, data_ch_time.shape[0]],
        vmin=vmin, vmax=vmax,
        cmap="RdBu_r"
    )
    plt.colorbar(label="Power")
    plt.xlabel("Time (s)")
    plt.ylabel("Channel index (0..63)")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# D) MAIN: subject → within-subject mean → group mean
group = {c: {b: [] for b in bands} for c in conds}

freqs_ref = None
times_ref = None
ch_ref = None

# time mask for plotting window
tmask_ref = None

errors = {}

for sub in subjects:
    try:
        X, ch_names, freqs, times, epochs = load_subject_data(sub)

        # lock references on first subject
        if freqs_ref is None:
            freqs_ref = freqs.copy()
            times_ref = times.copy()
            ch_ref = list(ch_names)
            tmask_ref = (times_ref >= TMIN) & (times_ref <= TMAX)
            if not np.any(tmask_ref):
                raise RuntimeError(f"Requested time window [{TMIN},{TMAX}] not in times range [{times_ref[0]},{times_ref[-1]}].")

        # consistency checks
        if not np.allclose(freqs, freqs_ref):
            raise RuntimeError("freqs mismatch vs first subject")
        if not np.allclose(times, times_ref):
            raise RuntimeError("times mismatch vs first subject")
        if list(ch_names) != ch_ref:
            raise RuntimeError("channel order mismatch vs first subject")

        if X.shape[0] != len(epochs):
            raise RuntimeError(f"trial count mismatch: X has {X.shape[0]} trials, epochs has {len(epochs)}")

        # per condition
        for cname, ckey in conds.items():
            idx = safe_cond_indices(epochs, ckey)
            if len(idx) == 0:
                raise RuntimeError(f"{sub}: no trials for condition {ckey}")

            # trial-mean in full TF: (ch,freq,time)
            M = X[idx].mean(axis=0)

            for bname, (f_lo, f_hi) in bands.items():
                D_ch_t = band_average_ch_time(M, freqs_ref, f_lo, f_hi)  # (ch,time)
                D_ch_t = D_ch_t[:, tmask_ref]  # crop to [-0.2,1]
                group[cname][bname].append(D_ch_t.astype(np.float32))

        print(f"[OK] {sub}")

    except Exception as e:
        errors[sub] = str(e)
        print(f"[SKIP] {sub} -> {e}")

print("\nKept per condition/band (should be same across all):")
for cname in conds:
    for bname in bands:
        print(cname, bname, len(group[cname][bname]))

if errors:
    print("\nErrors (first 5):")
    for k in list(errors.keys())[:5]:
        print(k, "->", errors[k])

#GROUP MEAN + PLOTS
#compute global vlim for comparable color scale across all plots (optional)
all_maps = []
for cname in conds:
    for bname in bands:
        if len(group[cname][bname]) > 0:
            all_maps.append(np.stack(group[cname][bname], axis=0).mean(axis=0))
all_maps = np.stack(all_maps, axis=0)
vmin, vmax = np.nanpercentile(all_maps, [5, 95])

times_plot = times_ref[tmask_ref]
ch_names_plot = ch_ref

for cname in conds:
    for bname, (f_lo, f_hi) in bands.items():
        X_stack = np.stack(group[cname][bname], axis=0)  # (subj,ch,time)
        mean_ch_t = X_stack.mean(axis=0)

        plot_heatmap_ch_time(
            mean_ch_t,
            times_plot,
            ch_names_plot,
            title=f"Group mean | {cname} | {bname} ({f_lo}-{f_hi} Hz) | {TMIN}..{TMAX}s",
            vlim=(vmin, vmax)
        )

In [ ]:
import numpy as np
import mne
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import os
os.environ["QT_API"] = "pyqt5"
%matplotlib qt

subjects = ["sub_m_1_02", "sub_m_1_06", "sub_m_1_10","sub_m_1_14","sub_m_1_18","sub_m_1_22","sub_m_1_26","sub_m_1_30","sub_m_1_34","sub_m_1_38","sub_m_1_42","sub_m_1_46",
            "sub_m_2_04","sub_m_2_08","sub_m_2_12","sub_m_2_16","sub_m_2_20","sub_m_2_24","sub_m_2_28","sub_m_2_32","sub_m_2_36","sub_m_2_40","sub_m_2_44",
            "sub_p_1_01","sub_p_1_05","sub_p_1_09","sub_p_1_13","sub_p_1_17","sub_p_1_21","sub_p_1_25","sub_p_1_29","sub_p_1_33","sub_p_1_35","sub_p_1_37","sub_p_1_41","sub_p_1_45",
            "sub_p_2_03","sub_p_2_07","sub_p_2_11","sub_p_2_15","sub_p_2_19","sub_p_2_27","sub_p_2_31","sub_p_2_39","sub_p_2_43","sub_p_2_47"]

TFR_DIR = r"F:/tfr_single_trial_cycle_8"
EPO_DIR = r"F:/epochs_ITC"

META_FNAME  = "{sub}_meta.npz"
POWER_FNAME = "{sub}_power.npy"
EPO_FNAME = "{sub}-raw-ica-reject-ERP-epo.fif"

conds = {"int_rep": "cue/cue1/int/rep", "int_swi": "cue/cue1/int/swi", "ext_rep": "cue/cue1/ext/rep","ext_swi": "cue/cue1/ext/swi"}
bands = {"Low Beta": (13, 16),"Beta": (16, 28),"alpha": (8, 15),"theta": (4, 7)}

TMIN, TMAX = -0.2, 1.0

def safe_cond_indices(epochs: mne.Epochs, cond_key: str):
    sel = epochs[cond_key].selection
    idx = np.flatnonzero(np.isin(epochs.selection, sel))
    return idx

def load_subject_data(sub):
    power_path = os.path.join(TFR_DIR, POWER_FNAME.format(sub=sub))
    meta_path  = os.path.join(TFR_DIR, META_FNAME.format(sub=sub))
    epo_path   = os.path.join(EPO_DIR,  EPO_FNAME.format(sub=sub))

    if not os.path.exists(power_path):
        raise FileNotFoundError(power_path)
    if not os.path.exists(meta_path):
        raise FileNotFoundError(meta_path)
    if not os.path.exists(epo_path):
        raise FileNotFoundError(epo_path)

    X = np.load(power_path, mmap_mode="r")  # (trial, ch, freq, time)
    meta = np.load(meta_path, allow_pickle=True)

    ch_names = meta["ch_names"].tolist() if hasattr(meta["ch_names"], "tolist") else list(meta["ch_names"])
    freqs = meta["freqs"].astype(float)
    times = meta["times"].astype(float)

    epochs = mne.read_epochs(epo_path, preload=False, verbose=False)

    # align epoch time to TFR time if needed (for safety)
    # Only needed for plotting mask; trial indexing is independent of time axis.
    return X, ch_names, freqs, times, epochs

def band_average_ch_time(M_ch_f_t, freqs, f_lo, f_hi):
    fmask = (freqs >= f_lo) & (freqs <= f_hi)
    if not np.any(fmask):
        raise RuntimeError(f"No freq bins inside [{f_lo},{f_hi}] Hz")
    return M_ch_f_t[:, fmask, :].mean(axis=1)  # (ch,time)

def plot_heatmap_ch_time(data_ch_time, times, ch_names, title, vlim=None):
    plt.figure(figsize=(10, 6))
    if vlim is None:
        vmin, vmax = np.nanpercentile(data_ch_time, [5, 95])
    else:
        vmin, vmax = vlim

    plt.imshow(
        data_ch_time,
        aspect="auto",
        origin="lower",
        extent=[times[0], times[-1], 0, data_ch_time.shape[0]],
        vmin=vmin, vmax=vmax,
        cmap="RdBu_r"
    )
    plt.colorbar(label="Power")
    plt.xlabel("Time (s)")
    plt.ylabel("Channel index (0..63)")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# D) MAIN: subject → within-subject mean → group mean
group = {c: {b: [] for b in bands} for c in conds}

freqs_ref = None
times_ref = None
ch_ref = None

# time mask for plotting window
tmask_ref = None

errors = {}

for sub in subjects:
    try:
        X, ch_names, freqs, times, epochs = load_subject_data(sub)

        # lock references on first subject
        if freqs_ref is None:
            freqs_ref = freqs.copy()
            times_ref = times.copy()
            ch_ref = list(ch_names)
            tmask_ref = (times_ref >= TMIN) & (times_ref <= TMAX)
            if not np.any(tmask_ref):
                raise RuntimeError(f"Requested time window [{TMIN},{TMAX}] not in times range [{times_ref[0]},{times_ref[-1]}].")

        # consistency checks
        if not np.allclose(freqs, freqs_ref):
            raise RuntimeError("freqs mismatch vs first subject")
        if not np.allclose(times, times_ref):
            raise RuntimeError("times mismatch vs first subject")
        if list(ch_names) != ch_ref:
            raise RuntimeError("channel order mismatch vs first subject")

        if X.shape[0] != len(epochs):
            raise RuntimeError(f"trial count mismatch: X has {X.shape[0]} trials, epochs has {len(epochs)}")

        # per condition
        for cname, ckey in conds.items():
            idx = safe_cond_indices(epochs, ckey)
            if len(idx) == 0:
                raise RuntimeError(f"{sub}: no trials for condition {ckey}")

            # trial-mean in full TF: (ch,freq,time)
            M = X[idx].mean(axis=0)

            for bname, (f_lo, f_hi) in bands.items():
                D_ch_t = band_average_ch_time(M, freqs_ref, f_lo, f_hi)  # (ch,time)
                D_ch_t = D_ch_t[:, tmask_ref]  # crop to [-0.2,1]
                group[cname][bname].append(D_ch_t.astype(np.float32))

        print(f"[OK] {sub}")

    except Exception as e:
        errors[sub] = str(e)
        print(f"[SKIP] {sub} -> {e}")

print("\nKept per condition/band (should be same across all):")
for cname in conds:
    for bname in bands:
        print(cname, bname, len(group[cname][bname]))

if errors:
    print("\nErrors (first 5):")
    for k in list(errors.keys())[:5]:
        print(k, "->", errors[k])

#GROUP MEAN + PLOTS
#compute global vlim for comparable color scale across all plots (optional)
all_maps = []
for cname in conds:
    for bname in bands:
        if len(group[cname][bname]) > 0:
            all_maps.append(np.stack(group[cname][bname], axis=0).mean(axis=0))
all_maps = np.stack(all_maps, axis=0)
vmin, vmax = np.nanpercentile(all_maps, [5, 95])

times_plot = times_ref[tmask_ref]
ch_names_plot = ch_ref

for cname in conds:
    for bname, (f_lo, f_hi) in bands.items():
        X_stack = np.stack(group[cname][bname], axis=0)  # (subj,ch,time)
        mean_ch_t = X_stack.mean(axis=0)

        plot_heatmap_ch_time(
            mean_ch_t,
            times_plot,
            ch_names_plot,
            title=f"Group mean | {cname} | {bname} ({f_lo}-{f_hi} Hz) | {TMIN}..{TMAX}s",
            vlim=(vmin, vmax)
        )

In [ ]:
#滑窗检查原始信号

import os
import numpy as np
import mne
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

EPOCHS_DIR = r"F:\epochs_ITC"
EPO_FNAME_PATTERN = "{sub}-raw-ica-reject-ERP-epo.fif"

subjects = ["sub_m_1_02", "sub_m_1_06", "sub_m_1_10","sub_m_1_14","sub_m_1_18","sub_m_1_22",
    "sub_m_1_26","sub_m_1_30","sub_m_1_34","sub_m_1_38","sub_m_1_42","sub_m_1_46",
    "sub_m_2_04","sub_m_2_08","sub_m_2_12","sub_m_2_16","sub_m_2_20","sub_m_2_24",
    "sub_m_2_28","sub_m_2_32","sub_m_2_36","sub_m_2_40","sub_m_2_44",
    "sub_p_1_01","sub_p_1_05","sub_p_1_09","sub_p_1_13","sub_p_1_17","sub_p_1_21",
    "sub_p_1_25","sub_p_1_29","sub_p_1_33","sub_p_1_35","sub_p_1_37","sub_p_1_41","sub_p_1_45",
    "sub_p_2_03","sub_p_2_07","sub_p_2_11","sub_p_2_15","sub_p_2_19","sub_p_2_27",
    "sub_p_2_31","sub_p_2_39","sub_p_2_43","sub_p_2_47"]

conds = {"int_rep": "cue/int/rep/cue2","int_swi": "cue/int/swi/cue2","ext_rep": "cue/ext/rep/cue2",
         "ext_swi": "cue/ext/swi/cue2"}

bands = {"theta (4–7)": (4, 7),"alpha (8–15)": (8, 15),"beta (15–28)": (15, 28)}

BASELINE = (-0.19921875, 0.0)
T_ANALYSIS = (-0.2, 1.0)

WIN_S = 0.25
STEP_S = 0.05

SFREQ_EXPECT = 308.0

# Welch PSD settings
FMIN = 4
FMAX = 28

EPS = 1e-20
cond_order = ["int_rep", "int_swi", "ext_rep", "ext_swi"]
cond_titles = {"int_rep": "INT / REP","int_swi": "INT / SWI","ext_rep": "EXT / REP","ext_swi": "EXT / SWI"}

def next_pow2(n: int) -> int:
    return 1 if n <= 1 else 2 ** int(np.ceil(np.log2(n)))

def time_to_index(epochs: mne.Epochs, t: float) -> int:
    return int(epochs.time_as_index(t)[0])

def bandpower_from_psd(psd, freqs, band):
    lo, hi = band
    mask = (freqs >= lo) & (freqs <= hi)
    if not np.any(mask):
        raise RuntimeError(f"No frequency bins inside band {band}.")
    return psd[..., mask].mean(axis=-1)

def compute_baseline_and_windows_db(epochs_cond: mne.Epochs, ch_names_ref=None):
    if epochs_cond.tmin > BASELINE[0] + 1e-12 or epochs_cond.tmax < T_ANALYSIS[1] - 1e-12:
        raise RuntimeError(
            f"Epochs time range [{epochs_cond.tmin:.3f}, {epochs_cond.tmax:.3f}] "
            f"does not cover baseline {BASELINE} and analysis {T_ANALYSIS}."
        )
    if ch_names_ref is not None:
        epochs_cond = epochs_cond.copy().pick_channels(ch_names_ref, ordered=True)

    sfreq = float(epochs_cond.info["sfreq"])

    data = epochs_cond.get_data()
    n_trials, n_ch, _ = data.shape

    b0 = time_to_index(epochs_cond, BASELINE[0])
    b1 = time_to_index(epochs_cond, BASELINE[1])
    if b1 <= b0:
        b1 = b0 + 1
    data_b = data[:, :, b0:b1]

    n_b = data_b.shape[-1]
    n_per_seg_b = n_b
    n_fft_b = next_pow2(n_per_seg_b)

    psd_b, freqs = mne.time_frequency.psd_array_welch(
        data_b,
        sfreq=sfreq,
        fmin=FMIN, fmax=FMAX,
        n_fft=n_fft_b,
        n_per_seg=n_per_seg_b,
        n_overlap=0,
        average="mean",
        verbose=False
    )

    baseline_band = {}
    for bname, brange in bands.items():
        bp = bandpower_from_psd(psd_b, freqs, brange)
        baseline_band[bname] = np.maximum(bp.mean(axis=0), EPS)

    t0, t1 = T_ANALYSIS
    starts = np.arange(t0, t1 - WIN_S + 1e-12, STEP_S)
    t_centers = starts + WIN_S / 2.0
    n_win = len(starts)

    out_db = {bname: np.zeros((n_ch, n_win), dtype=np.float64) for bname in bands.keys()}

    for wi, s in enumerate(starts):
        w0 = time_to_index(epochs_cond, s)
        w1 = time_to_index(epochs_cond, s + WIN_S)
        if w1 <= w0:
            w1 = w0 + 1
        data_w = data[:, :, w0:w1]

        n_w = data_w.shape[-1]
        n_per_seg_w = n_w
        n_fft_w = next_pow2(n_per_seg_w)

        psd_w, freqs_w = mne.time_frequency.psd_array_welch(
            data_w,
            sfreq=sfreq,
            fmin=FMIN, fmax=FMAX,
            n_fft=n_fft_w,
            n_per_seg=n_per_seg_w,
            n_overlap=0,
            average="mean",
            verbose=False
        )

        for bname, brange in bands.items():
            pw = bandpower_from_psd(psd_w, freqs_w, brange)
            pw_mean = np.maximum(pw.mean(axis=0), EPS)
            out_db[bname][:, wi] = 10.0 * np.log10(pw_mean / baseline_band[bname])

    return out_db, t_centers, epochs_cond.ch_names, sfreq

def choose_yticks(ch_names, step=1):
    idx = np.arange(0, len(ch_names), step)
    return idx, [ch_names[i] for i in idx]

group = {bname: {cname: [] for cname in conds.keys()} for bname in bands.keys()}
ch_ref = None
times_ref = None
sfreq_set = set()
skipped = {}
kept = 0

print("=== Sliding-window Welch PSD (baseline-referenced dB) ===")
print("Epochs dir:", EPOCHS_DIR)
print("Baseline :", BASELINE, "sec")
print("Analysis :", T_ANALYSIS, "sec")
print("Window   :", WIN_S, "sec   Step:", STEP_S, "sec")
print("Bands    :", bands)
print("--------------------------------------------------------")

for sub in subjects:
    try:
        epo_path = os.path.join(EPOCHS_DIR, EPO_FNAME_PATTERN.format(sub=sub))
        if not os.path.exists(epo_path):
            raise FileNotFoundError(epo_path)

        print(f"\n[{sub}] load epochs ...")
        epochs = mne.read_epochs(epo_path, preload=True, verbose=False)

        # EEG only
        epochs = epochs.copy().pick_types(eeg=True, eog=False, ecg=False, stim=False, misc=False)

        if ch_ref is None:
            ch_ref = epochs.ch_names

        # verify condition keys exist
        for cname, key in conds.items():
            if key not in epochs.event_id:
                raise KeyError(f"Condition key not in event_id: {key}")

        for cname, key in conds.items():
            epc = epochs[key]
            if len(epc) == 0:
                raise RuntimeError(f"No trials for {key}")

            out_db, t_centers, _, sfreq = compute_baseline_and_windows_db(epc, ch_names_ref=ch_ref)
            sfreq_set.add(float(sfreq))

            if times_ref is None:
                times_ref = t_centers
            else:
                if len(t_centers) != len(times_ref) or not np.allclose(t_centers, times_ref):
                    raise RuntimeError("Time-bin centers mismatch across subjects (sfreq or window params differ).")

            for bname in bands.keys():
                group[bname][cname].append(out_db[bname].astype(np.float32))

        kept += 1

    except Exception as e:
        skipped[sub] = str(e)
        print(f"[SKIP] {sub} -> {e}")

print("\n---------------- SUMMARY ----------------")
print("Kept subjects:", kept, "/", len(subjects))
print("Unique sfreq found:", sorted(list(sfreq_set)))
if skipped:
    print("Skipped:", len(skipped))
    for i, (k, v) in enumerate(skipped.items()):
        if i >= 8:
            break
        print(" ", k, "->", v)

if kept == 0:
    raise RuntimeError("No subjects kept. Fix paths / event_id keys first.")

# PLOT
for bname in bands.keys():
    avg = {}
    for cname in cond_order:
        mats = group[bname][cname]
        if len(mats) == 0:
            raise RuntimeError(f"No data for band={bname}, cond={cname}.")
        avg[cname] = np.stack(mats, axis=0).mean(axis=0)

    all_vals = np.concatenate([avg[c].ravel() for c in cond_order])
    vmin, vmax = np.nanpercentile(all_vals, [5, 95])

    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(
        f"{bname} — ΔPSD (dB) vs baseline {BASELINE[0]:.2f}..{BASELINE[1]:.2f}s  |  "
        f"analysis {T_ANALYSIS[0]:.1f}..{T_ANALYSIS[1]:.1f}s  |  win={WIN_S:.2f}s step={STEP_S:.2f}s",
        y=0.98
    )

    gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 0.05], wspace=0.25, hspace=0.25)
    axs = [
        fig.add_subplot(gs[0, 0]),
        fig.add_subplot(gs[0, 1]),
        fig.add_subplot(gs[1, 0]),
        fig.add_subplot(gs[1, 1]),
    ]
    cax = fig.add_subplot(gs[:, 2])

    x0, x1 = float(times_ref[0]), float(times_ref[-1])
    n_ch = avg[cond_order[0]].shape[0]

    yidx, ylab = choose_yticks(ch_ref, step=1)

    ims = []
    for ax, cname in zip(axs, cond_order):
        mat = avg[cname]
        im = ax.imshow(
            mat,
            aspect="auto",
            origin="lower",
            vmin=vmin, vmax=vmax,
            extent=[x0, x1, 0, n_ch],
            interpolation="nearest",
        )
        ims.append(im)
        ax.set_title(cond_titles[cname])
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Channel")
        ax.set_yticks(yidx)
        ax.set_yticklabels(ylab, fontsize=8)
        ax.axvline(0.0, color="k", linestyle="--", linewidth=1, alpha=0.7)

    cb = fig.colorbar(ims[0], cax=cax)
    cb.set_label("PSD (dB) relative to baseline")
    plt.show()

In [ ]:
#滑窗检查原始信号
import os
import numpy as np
import mne
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

EPOCHS_DIR = r"F:\epochs_ITC"
EPO_FNAME_PATTERN = "{sub}-raw-ica-reject-ERP-epo.fif"

subjects = ["sub_m_1_02", "sub_m_1_06", "sub_m_1_10","sub_m_1_14","sub_m_1_18","sub_m_1_22",
    "sub_m_1_26","sub_m_1_30","sub_m_1_34","sub_m_1_38","sub_m_1_42","sub_m_1_46",
    "sub_m_2_04","sub_m_2_08","sub_m_2_12","sub_m_2_16","sub_m_2_20","sub_m_2_24",
    "sub_m_2_28","sub_m_2_32","sub_m_2_36","sub_m_2_40","sub_m_2_44",
    "sub_p_1_01","sub_p_1_05","sub_p_1_09","sub_p_1_13","sub_p_1_17","sub_p_1_21",
    "sub_p_1_25","sub_p_1_29","sub_p_1_33","sub_p_1_35","sub_p_1_37","sub_p_1_41","sub_p_1_45",
    "sub_p_2_03","sub_p_2_07","sub_p_2_11","sub_p_2_15","sub_p_2_19","sub_p_2_27",
    "sub_p_2_31","sub_p_2_39","sub_p_2_43","sub_p_2_47"]

conds = {"int_rep": "targ/int/rep/sat","int_swi": "targ/int/swi/sat","ext_rep": "targ/ext/rep/sat",
         "ext_swi": "targ/ext/swi/sat"}

bands = {"theta (4–7)": (4, 7),"alpha (8–15)": (8, 15),"beta (15–28)": (15, 28)}

BASELINE = (0.8, 1.0)
T_ANALYSIS = (-0.2, 1.0)

WIN_S = 0.25
STEP_S = 0.05

SFREQ_EXPECT = 308.0

# Welch PSD settings
FMIN = 4
FMAX = 28

EPS = 1e-20
cond_order = ["int_rep", "int_swi", "ext_rep", "ext_swi"]
cond_titles = {"int_rep": "INT / REP","int_swi": "INT / SWI","ext_rep": "EXT / REP","ext_swi": "EXT / SWI"}

def next_pow2(n: int) -> int:
    return 1 if n <= 1 else 2 ** int(np.ceil(np.log2(n)))

def time_to_index(epochs: mne.Epochs, t: float) -> int:
    return int(epochs.time_as_index(t)[0])

def bandpower_from_psd(psd, freqs, band):
    lo, hi = band
    mask = (freqs >= lo) & (freqs <= hi)
    if not np.any(mask):
        raise RuntimeError(f"No frequency bins inside band {band}.")
    return psd[..., mask].mean(axis=-1)

def compute_baseline_and_windows_db(epochs_cond: mne.Epochs, ch_names_ref=None):
    if epochs_cond.tmin > BASELINE[0] + 1e-12 or epochs_cond.tmax < T_ANALYSIS[1] - 1e-12:
        raise RuntimeError(
            f"Epochs time range [{epochs_cond.tmin:.3f}, {epochs_cond.tmax:.3f}] "
            f"does not cover baseline {BASELINE} and analysis {T_ANALYSIS}."
        )
    if ch_names_ref is not None:
        epochs_cond = epochs_cond.copy().pick_channels(ch_names_ref, ordered=True)

    sfreq = float(epochs_cond.info["sfreq"])

    data = epochs_cond.get_data()
    n_trials, n_ch, _ = data.shape

    b0 = time_to_index(epochs_cond, BASELINE[0])
    b1 = time_to_index(epochs_cond, BASELINE[1])
    if b1 <= b0:
        b1 = b0 + 1
    data_b = data[:, :, b0:b1]

    n_b = data_b.shape[-1]
    n_per_seg_b = n_b
    n_fft_b = next_pow2(n_per_seg_b)

    psd_b, freqs = mne.time_frequency.psd_array_welch(
        data_b,
        sfreq=sfreq,
        fmin=FMIN, fmax=FMAX,
        n_fft=n_fft_b,
        n_per_seg=n_per_seg_b,
        n_overlap=0,
        average="mean",
        verbose=False
    )

    baseline_band = {}
    for bname, brange in bands.items():
        bp = bandpower_from_psd(psd_b, freqs, brange)
        baseline_band[bname] = np.maximum(bp.mean(axis=0), EPS)

    t0, t1 = T_ANALYSIS
    starts = np.arange(t0, t1 - WIN_S + 1e-12, STEP_S)
    t_centers = starts + WIN_S / 2.0
    n_win = len(starts)

    out_db = {bname: np.zeros((n_ch, n_win), dtype=np.float64) for bname in bands.keys()}

    for wi, s in enumerate(starts):
        w0 = time_to_index(epochs_cond, s)
        w1 = time_to_index(epochs_cond, s + WIN_S)
        if w1 <= w0:
            w1 = w0 + 1
        data_w = data[:, :, w0:w1]

        n_w = data_w.shape[-1]
        n_per_seg_w = n_w
        n_fft_w = next_pow2(n_per_seg_w)

        psd_w, freqs_w = mne.time_frequency.psd_array_welch(
            data_w,
            sfreq=sfreq,
            fmin=FMIN, fmax=FMAX,
            n_fft=n_fft_w,
            n_per_seg=n_per_seg_w,
            n_overlap=0,
            average="mean",
            verbose=False
        )

        for bname, brange in bands.items():
            pw = bandpower_from_psd(psd_w, freqs_w, brange)
            pw_mean = np.maximum(pw.mean(axis=0), EPS)
            out_db[bname][:, wi] = 10.0 * np.log10(pw_mean / baseline_band[bname])

    return out_db, t_centers, epochs_cond.ch_names, sfreq

def choose_yticks(ch_names, step=1):
    idx = np.arange(0, len(ch_names), step)
    return idx, [ch_names[i] for i in idx]

group = {bname: {cname: [] for cname in conds.keys()} for bname in bands.keys()}
ch_ref = None
times_ref = None
sfreq_set = set()
skipped = {}
kept = 0

print("=== Sliding-window Welch PSD (baseline-referenced dB) ===")
print("Epochs dir:", EPOCHS_DIR)
print("Baseline :", BASELINE, "sec")
print("Analysis :", T_ANALYSIS, "sec")
print("Window   :", WIN_S, "sec   Step:", STEP_S, "sec")
print("Bands    :", bands)
print("--------------------------------------------------------")

for sub in subjects:
    try:
        epo_path = os.path.join(EPOCHS_DIR, EPO_FNAME_PATTERN.format(sub=sub))
        if not os.path.exists(epo_path):
            raise FileNotFoundError(epo_path)

        print(f"\n[{sub}] load epochs ...")
        epochs = mne.read_epochs(epo_path, preload=True, verbose=False)

        # EEG only
        epochs = epochs.copy().pick_types(eeg=True, eog=False, ecg=False, stim=False, misc=False)

        if ch_ref is None:
            ch_ref = epochs.ch_names

        # verify condition keys exist
        for cname, key in conds.items():
            if key not in epochs.event_id:
                raise KeyError(f"Condition key not in event_id: {key}")

        for cname, key in conds.items():
            epc = epochs[key]
            if len(epc) == 0:
                raise RuntimeError(f"No trials for {key}")

            out_db, t_centers, _, sfreq = compute_baseline_and_windows_db(epc, ch_names_ref=ch_ref)
            sfreq_set.add(float(sfreq))

            if times_ref is None:
                times_ref = t_centers
            else:
                if len(t_centers) != len(times_ref) or not np.allclose(t_centers, times_ref):
                    raise RuntimeError("Time-bin centers mismatch across subjects (sfreq or window params differ).")

            for bname in bands.keys():
                group[bname][cname].append(out_db[bname].astype(np.float32))

        kept += 1

    except Exception as e:
        skipped[sub] = str(e)
        print(f"[SKIP] {sub} -> {e}")

print("\n---------------- SUMMARY ----------------")
print("Kept subjects:", kept, "/", len(subjects))
print("Unique sfreq found:", sorted(list(sfreq_set)))
if skipped:
    print("Skipped:", len(skipped))
    for i, (k, v) in enumerate(skipped.items()):
        if i >= 8:
            break
        print(" ", k, "->", v)

if kept == 0:
    raise RuntimeError("No subjects kept. Fix paths / event_id keys first.")

# PLOT
for bname in bands.keys():
    avg = {}
    for cname in cond_order:
        mats = group[bname][cname]
        if len(mats) == 0:
            raise RuntimeError(f"No data for band={bname}, cond={cname}.")
        avg[cname] = np.stack(mats, axis=0).mean(axis=0)

    all_vals = np.concatenate([avg[c].ravel() for c in cond_order])
    vmin, vmax = np.nanpercentile(all_vals, [5, 95])

    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(
        f"{bname} — ΔPSD (dB) vs baseline {BASELINE[0]:.2f}..{BASELINE[1]:.2f}s  |  "
        f"analysis {T_ANALYSIS[0]:.1f}..{T_ANALYSIS[1]:.1f}s  |  win={WIN_S:.2f}s step={STEP_S:.2f}s",
        y=0.98
    )

    gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 0.05], wspace=0.25, hspace=0.25)
    axs = [
        fig.add_subplot(gs[0, 0]),
        fig.add_subplot(gs[0, 1]),
        fig.add_subplot(gs[1, 0]),
        fig.add_subplot(gs[1, 1]),
    ]
    cax = fig.add_subplot(gs[:, 2])

    x0, x1 = float(times_ref[0]), float(times_ref[-1])
    n_ch = avg[cond_order[0]].shape[0]

    yidx, ylab = choose_yticks(ch_ref, step=1)

    ims = []
    for ax, cname in zip(axs, cond_order):
        mat = avg[cname]
        im = ax.imshow(
            mat,
            aspect="auto",
            origin="lower",
            vmin=vmin, vmax=vmax,
            extent=[x0, x1, 0, n_ch],
            interpolation="nearest",
        )
        ims.append(im)
        ax.set_title(cond_titles[cname])
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Channel")
        ax.set_yticks(yidx)
        ax.set_yticklabels(ylab, fontsize=8)
        ax.axvline(0.0, color="k", linestyle="--", linewidth=1, alpha=0.7)

    cb = fig.colorbar(ims[0], cax=cax)
    cb.set_label("PSD (dB) relative to baseline")
    plt.show()